In [19]:
import pandas as pd
import numpy as np

df_sales = pd.read_csv('../data/processed/clean_sales_2025.csv')
df_op_ex = pd.read_csv('../data/processed/clean_fixed_expenses_2025.csv')

display(df_op_ex)
display(df_sales.head())

,month,location_rent,nayax_subscription,sim_card_fee,zus
0,March,793.40,61.72,6.03,420.0
1,April,626.32,62.95,6.03,420.0
2,May,605.65,62.53,6.03,420.0
3,June,635.17,62.52,6.03,420.0
4,July,636.16,63.07,6.03,420.0
5,August,635.17,63.00,6.03,420.0
6,September,630.25,63.01,6.03,420.0
7,October,627.30,62.62,6.03,420.0
8,November,637.14,62.39,6.03,420.0
9,December,631.24,62.42,6.03,420.0


,transaction_id,price,beverage_name,card_brand,payment_method,transaction_time,month,day_of_week,hour,date,cogs,transaction_fee,gross_profit_per_cup
0,2617707542,7.0,Coffee with Milk,VISA,Karta bankowa(NFC),2025-12-31 10:43:15,December,Wednesday,10,2025-12-31,2.46,0.18,4.36
1,2617704370,6.0,Black Coffee,VISA,Karta bankowa(NFC),2025-12-31 10:41:07,December,Wednesday,10,2025-12-31,2.24,0.15,3.61
2,2616827380,7.0,Moccacino,MASTERCARD,Karta bankowa(NFC),2025-12-30 16:42:15,December,Tuesday,16,2025-12-30,3.30,0.18,3.52
3,6450508504,6.0,Espresso,VISA,Karta bankowa(CLS),2025-12-30 14:14:52,December,Tuesday,14,2025-12-30,1.22,0.15,4.63
4,2616674810,8.0,Cappuccino,VISA,Karta bankowa(NFC),2025-12-30 13:34:09,December,Tuesday,13,2025-12-30,2.67,0.20,5.13


In [50]:
###KPI's Calculations 

revenue = df_sales['price'].sum().round(2)

volume = len(df_sales)

cogs = df_sales['cogs'].sum().round(2)

sup_ex = round((total_volume * 0.20), 2)

trans_fees = df_sales['transaction_fee'].sum().round(2)
     
fixed_opex = df_op_ex[['location_rent', 'nayax_subscription', 'sim_card_fee', 'zus']].values.sum().round(2)

net_profit = (revenue - (cogs + trans_fees + fixed_opex + sup_ex)).round(2)

net_margin = round((net_profit / revenue), 2) * 100

# creating a dictionary with your calculated values
kpi_data = {
    'Metric': [
        'Total Revenue', 
        'Total Volume (Cups)', 
        'Total COGS', 
        'Total SUP Fees',
        'Total Transaction Fees', 
        'Total Fixed OpEx', 
        'Net Profit', 
        'Net Profit Margin'
    ],
    'Value': [
        f'{revenue:,.2f} PLN',
        f'{volume:,}',
        f'{cogs:,.2f} PLN',
        f'{sup_ex:,.2f} PLN',
        f'{trans_fees:,.2f} PLN',
        f'{fixed_opex:,.2f} PLN',
        f'{net_profit:,.2f} PLN',
        f'{net_margin:.2f}%'
    ]
}

# Convert to DataFrame and display
kpi_df = pd.DataFrame(kpi_data)
display(kpi_df)

,Metric,Value
0,Total Revenue,"24,889.14 PLN"
1,Total Volume (Cups),"3,604"
2,Total COGS,"8,952.25 PLN"
3,Total SUP Fees,720.80 PLN
4,Total Transaction Fees,626.80 PLN
5,Total Fixed OpEx,"11,344.33 PLN"
6,Net Profit,"3,244.96 PLN"
7,Net Profit Margin,13.00%


In [70]:
###Beverage Performance & Menu Analysis

product_summary = df_sales.groupby('beverage_name').agg(
    total_cups = ('transaction_id', 'count'),
    total_revenue = ('price', 'sum'),
    total_profit = ('gross_profit_per_cup', 'sum'),
    avg_price = ('price', 'mean')
).round(2).reset_index()

product_summary['profit_margin_ptc'] = ((product_summary['total_profit'] / product_summary['total_revenue']) * 100).round(2)
product_summary['volume_share_ptc'] = ((product_summary['total_cups'] / product_summary['total_cups'].sum()) * 100).round(2)

#sorting by total gross profit
ps_sorted_by_profit = product_summary.sort_values(by='total_profit', ascending=False)

#top 2 drinks by profit
display(ps_sorted_by_profit.head(2))

,beverage_name,total_cups,total_revenue,total_profit,avg_price,profit_margin_ptc,volume_share_ptc
1,Cappuccino,557,4456.0,2857.41,8.00,64.12,15.46
3,Coffee with Milk,734,4748.9,2823.80,6.47,59.46,20.37


In [93]:
###Temporal Analysis (Sales Patterns Over Time)

#Monthly Breakdown
mounthly_sales = df_sales.groupby('month').agg(
    total_cups = ('transaction_id', 'count'),
    total_revenue = ('price', 'sum')
).sort_values(by= 'total_revenue', ascending = False).reset_index()
display(mounthly_sales)

#Day of Week Breakdown
daily_revenue = df_sales.groupby(['date', 'day_of_week'], as_index = False)['price'].sum() # the revenue of each day
dow_sales = daily_revenue.groupby('day_of_week').agg(
    total_revenue = ('price', 'sum'),
    avg_revenue = ('price', 'mean')
).round(2).sort_values(by = 'avg_revenue', ascending = False).reset_index()
display(dow_sales)

# Hourly Traffic Breakdown
hourly_sales = df_sales.groupby('hour')['transaction_id'].count()
print(hourly_sales)

,month,total_cups,total_revenue
0,September,504,3517.13
1,August,455,3051.51
2,July,413,2777.95
3,November,375,2692.92
4,October,365,2627.37
5,March,345,2336.86
6,April,326,2224.94
7,June,308,2088.13
8,May,291,1980.39
9,December,222,1591.94


,day_of_week,total_revenue,avg_revenue
0,Friday,3889.44,94.86
1,Sunday,3874.46,94.50
2,Monday,3766.09,89.67
3,Tuesday,3724.62,88.68
4,Thursday,3522.09,85.90
5,Wednesday,3228.70,76.87
6,Saturday,2883.74,72.09


hour
0       3
2       1
3       1
4      20
5      41
6      76
7     204
8     270
9     402
10    493
11    368
12    352
13    348
14    358
15    264
16    149
17    117
18     56
19     51
20     23
21      2
22      2
23      3
Name: transaction_id, dtype: int64
